In [29]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from typing import TypedDict, Annotated
import operator


In [22]:
load_dotenv()

True

In [23]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

In [24]:
class EvaluationSchema(BaseModel):

    feedback:str = Field(description='Detailed feedback for the essay')
    score:int = Field(description='Score out of 10',ge  = 0,le = 10)
    

In [25]:
structured_model = model.with_structured_output(EvaluationSchema)

In [26]:
essay = """India, officially the Republic of India,[j][19] is a country in South Asia. It is the seventh-largest country by area, the most populous country in the world[20] and, since its independence in 1947, the world's most populous democracy.[21][22][23] Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land borders with Pakistan to the west;[k] China, Nepal and Bhutan to the north; Bangladesh and Myanmar to the east. In the Indian Ocean, India is near Sri Lanka and the Maldives. Its Andaman and Nicobar Islands share a maritime border with Myanmar, Thailand and Indonesia.

Modern humans arrived on the Indian subcontinent from Africa no later than 55,000 years ago.[25][26][27] Their long occupation, predominantly in isolation as hunter-gatherers, has made the region highly diverse.[28] Settled life emerged on the subcontinent in the western margins of the Indus river basin 9,000 years ago, evolving gradually into the Indus Valley Civilisation of the third millennium BCE.[29] By 1200 BCE, an archaic form of Sanskrit, an Indo-European language, had diffused into India from the northwest.[30][31] Its hymns recorded the early dawnings of Hinduism in India.[32] India's pre-existing Dravidian languages were supplanted in the northern regions.[33] By 400 BCE, caste had emerged within Hinduism,[34] and Buddhism and Jainism had arisen, proclaiming social orders unlinked to heredity.[35] Early political consolidations gave rise to the loose-knit Maurya and Gupta Empires.[36] During this era, there was a flourishing of creativity in art, architecture, and writing,[37] the status of women declined,[38] and untouchability became an organised belief.[l][39] In South India, the Middle kingdoms exported Dravidian language scripts and religious cultures to the kingdoms of Southeast Asia.[40]

In the 1st millennium Islam, Christianity, Judaism, and Zoroastrianism became established on India's southern and western coasts.[41] Early in the 2nd millennium Muslim armies from Central Asia intermittently overran India's northern plains.[42] The resulting Delhi Sultanate drew northern India into the cosmopolitan networks of medieval Islam.[43] In south India, the Vijayanagara Empire created a long-lasting composite Hindu culture.[44] In the Punjab, Sikhism emerged, rejecting institutionalised religion.[45] The Mughal Empire ushered in two centuries of economic expansion and relative peace,[46] and left a rich architectural legacy.[47][48] Gradually expanding rule of the British East India Company turned India into a colonial economy but consolidated its sovereignty.[49] British Crown rule began in 1858. The rights promised to Indians were granted slowly,[50][51] but technological changes were introduced, and modern ideas of education and the public life took root.[52] A nationalist movement emerged in India, the first in the non-European British Empire and an influence on other nationalist movements.[53][54] Noted for nonviolent resistance after 1920,[55] it became the primary factor in ending British rule.[56] In 1947, the British Indian Empire was partitioned into two independent dominions, a Hindu-majority dominion of India and a Muslim-majority dominion of Pakistan. A large-scale loss of life and an unprecedented migration accompanied the partition.[57]

India has been a federal republic since 1950, governed through a democratic parliamentary system. It is a pluralistic, multilingual and multi-ethnic society. India's population grew from 361 million in 1951 to over 1.4 billion in 2023.[58] During this time, its nominal per capita income increased from US$64 annually to US$2,601, and its literacy rate from 16.6% to 74%.[59] The Indian economy has since become a fast-growing major economy and a hub for information technology, with an expanding middle class.[60] India has reduced its poverty rate, though at the cost of increasing economic inequality.[61] It is a nuclear-weapon state that ranks high in military expenditure. It has disputes over Kashmir with its neighbours, Pakistan and China, unresolved since the mid-20th century.[62] Among the socio-economic challenges India faces are gender inequality, child malnutrition,[63] and rising levels of air pollution.[64] India's land is megadiverse with four biodiversity hotspots. India's wildlife, which has traditionally been viewed with tolerance in its culture, is supported in protected habitats.[65]

Etymology
Main article: Names for India
According to the Oxford English Dictionary, the English proper noun "India" derives most immediately from the Classical Latin India, a reference to a loosely-defined historical region of Asia stretching from South Asia to the borders of China. Further etymons are: Hellenistic Greek India (Ἰνδία); Ancient Greek Indos (Ἰνδός), or the River Indus; Achaemenian Old Persian Hindu (an eastern province of the Achaemenid Empire); and Sanskrit Sindhu, or "river," but specifically the Indus river, and by extension its well-settled basin.[66] The Ancient Greeks referred to South Asians as Indoi, 'the people of the Indus'.[67]

The term Bharat (Bhārat; pronounced [ˈbʱaːɾət] ⓘ), mentioned in both Indian epic poetry and the Constitution of India,[68][69] is used in its variations by many Indian languages. A modern rendering of the historical name Bharatavarsha, which applied originally to North India,[70][71] Bharat gained increased currency from the mid-19th century as a native name for India.[68][72]

Hindustan ([ɦɪndʊˈstaːn] ⓘ) is a Middle Persian name for India that became popular by the 13th century,[73] and was used widely since the era of the Mughal Empire. The meaning of Hindustan has varied, referring to a region encompassing the northern Indian subcontinent (present-day northern India and Pakistan) or to India in its near entirety.[68][72][74]"""

In [ ]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n. {essay}'
response = structured_model.invoke(prompt)
print(response)

feedback='The language quality of this text is exceptionally high. It demonstrates excellent clarity, precision, and a sophisticated vocabulary, characteristic of a well-researched and professionally written encyclopedic entry. The grammar and syntax are flawless, with varied sentence structures that contribute to the readability and flow of information. The organization is logical, presenting a comprehensive historical and geographical overview of India in a clear and concise manner. The tone is consistently objective and informative, making it an authoritative piece of writing. The use of specific terminology is accurate and appropriate throughout.' score=9


In [32]:
class UPSCState(TypedDict):
    
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[str],operator.add]
    avg_score:float

In [35]:
def evaluate_language(state:UPSCState):
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n. {state['essay']}'
    output = structured_model.invoke(prompt)


    return {'language_feedback':output.feedback,'individual_scores':[output.score]}
    

In [36]:
def evaluate_analysis(state:UPSCState):
    prompt = f'Evaluate the depth of the analysis of the following essay and provide a feedback and assign a score out of 10 \n. {state['essay']}'

    output = structured_model.invoke(prompt)


    return {'evaluate_feedback':output.feedback,'individual_scores':[output.score]}

In [37]:
def evaluate_thought(state:UPSCState):
    prompt = f'Evaluate the clarity of thoughts of the following essay and provide a feedback and assign a score out of 10 \n. {state['essay']}'

    output = structured_model.invoke(prompt)


    return {'clarity_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def final_evaluation(state:UPSCState):
    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}
    
    # avg calculation



_IncompleteInputError: incomplete input (3999372920.py, line 4)

In [38]:
graph = StateGraph(UPSCState)

# adding nodes
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)


NameError: name 'final_evaluation' is not defined